In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT.parent / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from metrics_group_merger import MetricsGroup, PCMDIRun, merge_metrics_group
from synthetic_metrics_workflow import (
    dataset_for_merged_group,
    make_comparison_parameters,
    run_synthetic_plots,
)


In [ ]:
# Plot setup: optionally merge two model groups, then compare them.
CASE_ID = "v20260212"
TITLE_LABEL = "Near Future versus Historical"

# Set RUN_MERGE=True when raw run JSONs should be merged before plotting.
RUN_MERGE = False
CLEAN_MERGED_OUTPUT = False
DRY_RUN_MERGE = False

# Merged metrics location and local config.
DIAG_ROOT = Path("/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le")
METRICS_DATA_ROOT = Path("/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data")
METRIC_CONFIG_FILE = PROJECT_ROOT / "config" / "synthetic_metrics_list.json"
OUTPUT_DIR = DIAG_ROOT / "Future_Metrics"

MIP = "e3sm"
EXP = "historical"
COMMON_MOVS_MODES = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
COMMON_MOVS_OBSES = (
    "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,"
    "HadISST,HadISST,HadISST"
)

# Raw runs to merge for each comparison group.
REF_GROUP_LABEL = "E3SMv3-Historical"
REF_MERGED_GROUP = "historical"
REF_RUNS = [
    PCMDIRun(
        case="v3.LR.historical_0051",
        model_name="e3sm.historical.v3-LR_0051",
        www=DIAG_ROOT / "hist",
    ),
]
REF_RAW_GROUP = MetricsGroup(
    name=REF_MERGED_GROUP,
    model_pattern="*",
    runs=REF_RUNS,
    mips=[MIP],
    exps=[EXP],
    case_id=CASE_ID,
    model_rename=("v3-LR", "v3-LR-Historical", True),
    movs_years=(1980, 2014),
    movs_modes=COMMON_MOVS_MODES,
    movs_obses=COMMON_MOVS_OBSES,
    enso_collections="ENSO_perf,ENSO_tel,ENSO_proc",
    enso_obses="Tropflux,Tropflux,Tropflux",
)

TEST_GROUP_LABEL = "E3SMv3-Future"
TEST_MERGED_GROUP = "future"
TEST_RUNS = [
    PCMDIRun(
        case="v3.LR.historical_0051",
        model_name="e3sm.historical.v3-LR_0051",
        www=DIAG_ROOT / "future",
    ),
]
TEST_RAW_GROUP = MetricsGroup(
    name=TEST_MERGED_GROUP,
    model_pattern="*",
    runs=TEST_RUNS,
    mips=[MIP],
    exps=[EXP],
    case_id=CASE_ID,
    model_rename=("v3-LR", "v3-LR-Future", True),
    movs_years=(2015, 2049),
    movs_modes=COMMON_MOVS_MODES,
    movs_obses=COMMON_MOVS_OBSES,
    enso_collections="ENSO_perf,ENSO_tel,ENSO_proc",
    enso_obses="FAKE-FUTURE2,FAKE-FUTURE2,FAKE-FUTURE2",
)

if RUN_MERGE:
    for raw_group in [REF_RAW_GROUP, TEST_RAW_GROUP]:
        merge_metrics_group(
            raw_group,
            metrics_root=METRICS_DATA_ROOT,
            run_type="model_vs_obs",
            enable_clim=True,
            enable_movs=True,
            enable_enso=True,
            strict=True,
            verbose=True,
            dry_run=DRY_RUN_MERGE,
            clean=CLEAN_MERGED_OUTPUT,
        )

# Merged groups consumed by the plot workflow.
REF_DATASET = dataset_for_merged_group(
    mip=MIP,
    group=REF_MERGED_GROUP,
    case_id=CASE_ID,
    root=METRICS_DATA_ROOT,
)
TEST_DATASET = dataset_for_merged_group(
    mip=MIP,
    group=TEST_MERGED_GROUP,
    case_id=CASE_ID,
    root=METRICS_DATA_ROOT,
)
ALIGN_WITH_CMIP = True

# Diagnostics to generate.
CLIM_VIEWER = False
MOVA_VIEWER = False
MOVC_VIEWER = False
ENSO_VIEWER = True

# Plot settings.
CLIM_VARS = (
    "pr,prw,psl,rlds,rldscs,rltcre,rstcre,rsus,rsuscs,rlus,rlut,rlutcs,"
    "rsds,rsdscs,rsut,rsutcs,rtmt,sfcWind,tas,tauu,tauv,ts,ta-200,ta-850,"
    "ua-200,ua-850,va-200,va-850,zg-500"
)
CLIM_REGIONS = "global"
MOVS_GROUP = "cbf"
ATM_MODES = None
ATM_OBS = "NOAA-20C"
CPL_MODES = None
CPL_OBS = "HadISST"
ERROR_NORM = "reference"
FIGURE_FORMAT = "pdf"

EXCLUDE_VARS = {
    "E3SM-1-0": ["ta-850"],
    "E3SM-1-1-ECA": ["ta-850"],
    "CIESM": ["pr"],
    "KIOST-ESM": ["zg-500", "ta-850"],
    "GISS-E2-2-G": ["rlutcs", "zg-500"],
}
EXCLUDE_MODELS = ["E3SM-1-0", "E3SM-1-1", "E3SM-1-1-ECA", "E3SM-2-0", "E3SM-2-1"]

parameters = make_comparison_parameters(
    case_id=CASE_ID,
    ref_group=REF_GROUP_LABEL,
    test_group=TEST_GROUP_LABEL,
    ref_dataset=REF_DATASET,
    test_dataset=TEST_DATASET,
    metrics_root=METRICS_DATA_ROOT,
    out_dir=OUTPUT_DIR,
    align_with_cmip=ALIGN_WITH_CMIP,
    test_combined=True,
    test_model_only=False,
    clim_vars=CLIM_VARS,
    clim_regions=CLIM_REGIONS,
    movs_group=MOVS_GROUP,
    error_norm=ERROR_NORM,
    exclude_vars=EXCLUDE_VARS,
    exclude_models=EXCLUDE_MODELS,
    atm_modes=ATM_MODES,
    atm_obs=ATM_OBS,
    cpl_modes=CPL_MODES,
    cpl_obs=CPL_OBS,
    figure_format=FIGURE_FORMAT,
    clim_viewer=CLIM_VIEWER,
    mova_viewer=MOVA_VIEWER,
    movc_viewer=MOVC_VIEWER,
    enso_viewer=ENSO_VIEWER,
)

run_synthetic_plots(
    parameters,
    title_label=TITLE_LABEL,
    metric_file=METRIC_CONFIG_FILE,
    debug=True,
)
